In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, TypedDict
import operator
from langgraph.checkpoint.memory import InMemorySaver

c:\rachna\test\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Build the graph
class State(TypedDict):
    data: Annotated[str, operator.add]   

def node_a(state: State) -> State:
    print("Node A executed")
    return {"data": "A "}

def node_b(state: State) -> State:
    print("Node B executed")
    return {"data": "B "}

def node_c(state: State) -> State:
    print("Node C executed")
    return {"data": "C "}

def node_d(state: State) -> State:
    print("Node D executed")
    return {"data": "D "}

def node_e(state: State) -> State:
    print("Node E executed")
    return {"data": "E "}

graph_builder = StateGraph(State)

graph_builder.add_node("A", node_a)
graph_builder.add_node("B", node_b)
graph_builder.add_node("C", node_c)
graph_builder.add_node("D", node_d)
graph_builder.add_node("E", node_e)

graph_builder.add_edge(START, "A")
graph_builder.add_edge("A", "B")
graph_builder.add_edge("B", "C")
graph_builder.add_edge("B", "D")
graph_builder.add_edge("C", "E")
graph_builder.add_edge("D", "E")
graph_builder.add_edge("E", END)

In [3]:
# Create an InMemorySaver checkpointer
memory = InMemorySaver()

# Compile the graph with the checkpointer enabled
graph = graph_builder.compile(checkpointer=memory)

In [4]:
# # Define a runtime configuration with a thread_id
config = {"configurable": {"thread_id": "demo-thread-1"}}

# Invoke the graph with config
final_state = graph.invoke({"data": ""}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E '}


In [5]:
# Retrieve the latest checkpoint (state snapshot) for this thread
graph.get_state(config)

StateSnapshot(values={'data': 'A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a86-6ac8-8004-e003d196f479'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T22:58:56.551802+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a84-642d-8003-bab3fab15aad'}}, tasks=(), interrupts=())

In [5]:
# Access only the state values from the latest checkpoint
graph.get_state(config).values

{'data': 'A B C D E '}

In [7]:
# Retrieve the full checkpoint history for this thread
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a86-6ac8-8004-e003d196f479'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T22:58:56.551802+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a84-642d-8003-bab3fab15aad'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a84-642d-8003-bab3fab15aad'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T22:58:56.550814+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb534-7a81-6ce5-8002-bbdea82e1354'}}, tasks=(PregelTask(id='062ac862-7e72-863d-0815-872541656907', name='E', path=('__pregel_pull', 'E'), erro

In [8]:
# Invoke the graph again using the SAME thread_id, workflow resumes from the last checkpoint
config = {"configurable": {"thread_id": "demo-thread-1"}}
final_state = graph.invoke({"data": "New "}, config=config)
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': 'A B C D E New A B C D E '}


In [9]:
# View the updated checkpoint history, this now includes checkpoints from both executions
list(graph.get_state_history(config))

[StateSnapshot(values={'data': 'A B C D E New A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb53f-9b8b-68e5-800a-88180118be4d'}}, metadata={'source': 'loop', 'step': 10, 'parents': {}}, created_at='2026-01-06T23:03:55.293104+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb53f-9b8b-68e4-8009-e3be32e6203b'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': 'A B C D E New A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb53f-9b8b-68e4-8009-e3be32e6203b'}}, metadata={'source': 'loop', 'step': 9, 'parents': {}}, created_at='2026-01-06T23:03:55.293104+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb53f-9b86-6afd-8008-a6c25699d8c8'}}, tasks=(PregelTask(id='38a8901d-24b3-3be9-c6c2-721996502d8e', name='E', path

In [10]:
# Invoke the graph with a DIFFERENT thread_id, This creates a completely new, independent execution
final_state = graph.invoke({"data": "222 "}, config={"configurable": {"thread_id": "demo-thread-2"}})
print("Final state:", final_state)

Node A executed
Node B executed
Node C executed
Node D executed
Node E executed
Final state: {'data': '222 A B C D E '}


In [11]:
# Retrieve checkpoint history for demo-thread-2, this is separate from demo-thread-1
list(graph.get_state_history(config={"configurable": {"thread_id": "demo-thread-2"}}))

[StateSnapshot(values={'data': '222 A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6ace-8004-4019d1df96ec'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T23:05:09.870254+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6acd-8003-802eb127f677'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'data': '222 A B C D '}, next=('E',), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6acd-8003-802eb127f677'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-01-06T23:05:09.870254+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c2-63e0-8002-83136b8de330'}}, tasks=(PregelTask(id='a5141eec-e9e2-bbbc-f3eb-e1b2d15cbef3', name='E', path=('__pregel_pull', 'E

In [12]:
# Get the latest state snapshot for demo-thread-2
config = {"configurable": {"thread_id": "demo-thread-2"}}
graph.get_state(config)

StateSnapshot(values={'data': '222 A B C D E '}, next=(), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6ace-8004-4019d1df96ec'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='2026-01-06T23:05:09.870254+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62c4-6acd-8003-802eb127f677'}}, tasks=(), interrupts=())

In [13]:
# Retrieve a specific checkpoint using checkpoint_id, this allows inspection of state at an exact execution point
config = {"configurable": {"thread_id": "demo-thread-2", "checkpoint_id": '1f0eb542-62c2-63e0-8002-83136b8de330'}}
graph.get_state(config)

StateSnapshot(values={'data': '222 A B '}, next=('C', 'D'), config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_id': '1f0eb542-62c2-63e0-8002-83136b8de330'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-01-06T23:05:09.869257+00:00', parent_config={'configurable': {'thread_id': 'demo-thread-2', 'checkpoint_ns': '', 'checkpoint_id': '1f0eb542-62bf-6ce6-8001-47bd60bd7ade'}}, tasks=(PregelTask(id='2ea3b500-10f0-e938-dafa-531e9f53c513', name='C', path=('__pregel_pull', 'C'), error=None, interrupts=(), state=None, result={'data': 'C '}), PregelTask(id='c7445894-71f8-9f89-e402-f5402ae3592b', name='D', path=('__pregel_pull', 'D'), error=None, interrupts=(), state=None, result={'data': 'D '})), interrupts=())